# Tokenization process breakdown


## Translation of strings to Unicode
In python strings are immutable sequences of Unicode code points 

Its basically a way to define lots of diffrent characters. Currently there are 150,000 characters. 



In [1]:
# in python to get the unicode of a char
ord("h")

104

In [2]:
ord("😊")

128522

In [9]:
[ord(x) for x in "안녕하세요 👋 (hello in Korean!"]

[50504,
 45397,
 54616,
 49464,
 50836,
 32,
 128075,
 32,
 40,
 104,
 101,
 108,
 108,
 111,
 32,
 105,
 110,
 32,
 75,
 111,
 114,
 101,
 97,
 110,
 33]

There are 3 types of encoding standarts UTF8, UTF16 and UTF32. These encodings are a way for us to take unicode text and get thier binary representaion (byte strings).

UTF8 takes every single code point and it translates it to a byte stream and these byte streams are between 1 to 4 bytes. https://utf8everywhere.org/

In [10]:
"안녕하세요 👋 (hello in Korean!".encode("utf-8")

b'\xec\x95\x88\xeb\x85\x95\xed\x95\x98\xec\x84\xb8\xec\x9a\x94 \xf0\x9f\x91\x8b (hello in Korean!'

In [14]:
# here we will get the raw bytes
list("안녕하세요 👋 (hello in Korean!".encode("utf-8"))

[236,
 149,
 136,
 235,
 133,
 149,
 237,
 149,
 152,
 236,
 132,
 184,
 236,
 154,
 148,
 32,
 240,
 159,
 145,
 139,
 32,
 40,
 104,
 101,
 108,
 108,
 111,
 32,
 105,
 110,
 32,
 75,
 111,
 114,
 101,
 97,
 110,
 33]

In [15]:
list("안녕하세요 👋 (hello in Korean!".encode("utf-16"))

[255,
 254,
 72,
 197,
 85,
 177,
 88,
 213,
 56,
 193,
 148,
 198,
 32,
 0,
 61,
 216,
 75,
 220,
 32,
 0,
 40,
 0,
 104,
 0,
 101,
 0,
 108,
 0,
 108,
 0,
 111,
 0,
 32,
 0,
 105,
 0,
 110,
 0,
 32,
 0,
 75,
 0,
 111,
 0,
 114,
 0,
 101,
 0,
 97,
 0,
 110,
 0,
 33,
 0]

In [16]:
list("안녕하세요 👋 (hello in Korean!".encode("utf-32"))

[255,
 254,
 0,
 0,
 72,
 197,
 0,
 0,
 85,
 177,
 0,
 0,
 88,
 213,
 0,
 0,
 56,
 193,
 0,
 0,
 148,
 198,
 0,
 0,
 32,
 0,
 0,
 0,
 75,
 244,
 1,
 0,
 32,
 0,
 0,
 0,
 40,
 0,
 0,
 0,
 104,
 0,
 0,
 0,
 101,
 0,
 0,
 0,
 108,
 0,
 0,
 0,
 108,
 0,
 0,
 0,
 111,
 0,
 0,
 0,
 32,
 0,
 0,
 0,
 105,
 0,
 0,
 0,
 110,
 0,
 0,
 0,
 32,
 0,
 0,
 0,
 75,
 0,
 0,
 0,
 111,
 0,
 0,
 0,
 114,
 0,
 0,
 0,
 101,
 0,
 0,
 0,
 97,
 0,
 0,
 0,
 110,
 0,
 0,
 0,
 33,
 0,
 0,
 0]

Here we can see that with UTF16 and 32 encodings we are getting alot of 0. Its very wastefull and not very desirable. Meaning that we want to stick with UTF8.

However we do NOT want to use it naivley. The reason being is because it would imply a vocab length of 256 possible tokens. If we **only** use UTF8 this will cause our text to be streched out over very very long seqeunces of bytes. This will make the embedding table very tiny (256) and the prediction layer will be tiny but the sequences will be very long.

We will have a finite context length but have very long sequences making it very inefficent for the next token prediction task. 

**Main idea**: we do not want to use the UTF8 by itself. We want to use a larger vocab size that we could tune as a hyper parameter but we want to stick with the UTF8 encoding of these strings

**Perfect idea**: It would be very nice to feed the raw bytes to the transformer. However for that we would need to change the transformer architecture. The reason is becuase the attention will become very expensive due to the very long sequences. 

## Byte Per Encoding (BPE) Algorithm

The BPE algorithm will allow us to compress these byte sequences to a variable amount  (https://en.wikipedia.org/wiki/Byte-pair_encoding). 

The main idea of the algorithm is as follows:

1. We get the input sequence: aaabdaaabac (as we can see the vocab size here is 4: 'a', 'b', 'c', 'd')

2. We itterate over the string and we find the pair of tokens that accour most frequently. Once we identify that pair we replace it with a single new token.

   Z = 'aa' -> ZabdZabac

3. Repeat

   Y = 'ab' -> ZYdZYac

   X = 'ZY' -> XdXac

4. Final:

   Z = 'aa'

   Y = 'ab'

   X = 'ZY'

   **Final**: XdXac


As we can see the vocab after BPE is 6 however the seqeunce size is 5. Compared to the previous (before BPE) where the vocab was 4 but the sequence was 11. 

Basically in the begining we had 256 vocab words (due to UTF8). BPE will find pairs of chars and combine them into one token and appand them to then vocab. This increases the size of the vocab **BUT** decreases the sequence length.

### Implementing BPE

**Step 0**: get the text and convert it into bytes. 

We can see that the tokens(the bytes for now) are much longer than the chars. Its becuase the wierd chars encode with more bytes than regular chars

In [19]:
text = "Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception."
# get the raww bytes
tokens = text.encode("utf-8")
# convert to a list of integers in range from 0-255
tokens = list(map(int, tokens))
print("-----------------------------------------------")
print(text)
print("Length:", len(text))
print("-----------------------------------------------")
print(tokens)
print("Length:", len(tokens))

-----------------------------------------------
Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception.
Length: 533
-----------------------------------------------
[239, 188, 181, 239, 189, 142, 239, 189, 137, 239, 189, 131, 239, 189, 143, 239, 189, 132, 239, 189, 133, 33, 32, 240, 159, 133, 164, 240, 159, 133, 157, 240, 159, 133, 152, 240, 159, 133, 146, 240, 159, 133, 158, 240, 159, 133, 147, 240, 159, 133, 148, 226, 128, 189, 32, 240, 159, 135, 186, 226, 128, 140, 240, 159, 135, 179, 226, 128, 140, 240, 159, 135, 

**Step 2**: Iterate over the bytes and find a pair of bytes that happen mopst frequently

In [38]:
def get_stats(ids):
    counts = {}

    for token_index in range(len(ids) - 1):
        current_pair = (ids[token_index], ids[token_index + 1])
        counts[current_pair] = counts.get(current_pair, 0) + 1

    return counts

stats = get_stats(tokens)
print(stats)
print("---------------- SORTED ----------------")
print(sorted(((v,k) for k,v in stats.items()), reverse=True))

{(239, 188): 1, (188, 181): 1, (181, 239): 1, (239, 189): 6, (189, 142): 1, (142, 239): 1, (189, 137): 1, (137, 239): 1, (189, 131): 1, (131, 239): 1, (189, 143): 1, (143, 239): 1, (189, 132): 1, (132, 239): 1, (189, 133): 1, (133, 33): 1, (33, 32): 2, (32, 240): 3, (240, 159): 15, (159, 133): 7, (133, 164): 1, (164, 240): 1, (133, 157): 1, (157, 240): 1, (133, 152): 1, (152, 240): 1, (133, 146): 1, (146, 240): 1, (133, 158): 1, (158, 240): 1, (133, 147): 1, (147, 240): 1, (133, 148): 1, (148, 226): 1, (226, 128): 12, (128, 189): 1, (189, 32): 1, (159, 135): 7, (135, 186): 1, (186, 226): 1, (128, 140): 6, (140, 240): 6, (135, 179): 1, (179, 226): 1, (135, 174): 1, (174, 226): 1, (135, 168): 1, (168, 226): 1, (135, 180): 1, (180, 226): 1, (135, 169): 1, (169, 226): 1, (135, 170): 1, (170, 33): 1, (159, 152): 1, (152, 132): 1, (132, 32): 1, (32, 84): 1, (84, 104): 1, (104, 101): 6, (101, 32): 20, (32, 118): 1, (118, 101): 3, (101, 114): 6, (114, 121): 2, (121, 32): 2, (32, 110): 2, (110,

**Step 3**: Now we have identified the most common pair. We will itterate over the sequence we will mint a new token ID of 256 (remember UTF8 is from 0 to 255) and everytime that we see the most common bit pair we will replace it with 256.

In [33]:
# call max on the dictionary
top_pair = max(stats, key=stats.get)

def merge(old_tokens, pair, idx):
    new_tokens = []
    skip = False
    for token_index in range(len(old_tokens)):
        if token_index == len(old_tokens) - 1:
            new_tokens.append(old_tokens[token_index])
            break
        
        elif skip == True:
            skip = False

        elif old_tokens[token_index] == pair[0] and old_tokens[token_index + 1] == pair[1]:
            new_tokens.append(idx)
            skip = True

        else:
            new_tokens.append(old_tokens[token_index])

    return new_tokens

tokens2 = merge(tokens, top_pair, 256)
print(tokens2)
print("Length: ", len(tokens2))

[239, 188, 181, 239, 189, 142, 239, 189, 137, 239, 189, 131, 239, 189, 143, 239, 189, 132, 239, 189, 133, 33, 32, 240, 159, 133, 164, 240, 159, 133, 157, 240, 159, 133, 152, 240, 159, 133, 146, 240, 159, 133, 158, 240, 159, 133, 147, 240, 159, 133, 148, 226, 128, 189, 32, 240, 159, 135, 186, 226, 128, 140, 240, 159, 135, 179, 226, 128, 140, 240, 159, 135, 174, 226, 128, 140, 240, 159, 135, 168, 226, 128, 140, 240, 159, 135, 180, 226, 128, 140, 240, 159, 135, 169, 226, 128, 140, 240, 159, 135, 170, 33, 32, 240, 159, 152, 132, 32, 84, 104, 256, 118, 101, 114, 121, 32, 110, 97, 109, 256, 115, 116, 114, 105, 107, 101, 115, 32, 102, 101, 97, 114, 32, 97, 110, 100, 32, 97, 119, 256, 105, 110, 116, 111, 32, 116, 104, 256, 104, 101, 97, 114, 116, 115, 32, 111, 102, 32, 112, 114, 111, 103, 114, 97, 109, 109, 101, 114, 115, 32, 119, 111, 114, 108, 100, 119, 105, 100, 101, 46, 32, 87, 256, 97, 108, 108, 32, 107, 110, 111, 119, 32, 119, 256, 111, 117, 103, 104, 116, 32, 116, 111, 32, 226, 128, 156

**Step 4**: Repeat steps 2-3 until all of the similar pairs have been converted. The question now how long do we do it. Its up to us, the longer that we do it the bigger the vocab will be but the smaller the sequence is going to be. This is a hyper parametr, we tune it as we go for the best results.

In [39]:
vocab_size = 276 # the desired final vocabulary size
num_merges = vocab_size - 256
ids = list(tokens) # copy so we don't destroy the original list

merges = {} # (int, int) -> int
for i in range(num_merges):
  stats = get_stats(ids)
  pair = max(stats, key=stats.get)
  idx = 256 + i
  print(f"merging {pair} into a new token {idx}")
  ids = merge(ids, pair, idx)
  merges[pair] = idx


merging (101, 32) into a new token 256
merging (240, 159) into a new token 257
merging (226, 128) into a new token 258
merging (105, 110) into a new token 259
merging (115, 32) into a new token 260
merging (97, 110) into a new token 261
merging (116, 104) into a new token 262
merging (257, 133) into a new token 263
merging (257, 135) into a new token 264
merging (97, 114) into a new token 265
merging (239, 189) into a new token 266
merging (258, 140) into a new token 267
merging (267, 264) into a new token 268
merging (101, 114) into a new token 269
merging (111, 114) into a new token 270
merging (116, 32) into a new token 271
merging (259, 103) into a new token 272
merging (115, 116) into a new token 273
merging (261, 100) into a new token 274
merging (32, 262) into a new token 275


The Tokenizer is a completly diffrent object from the LLM. It has its own training dataset of text on which you train the BPE. 

In a state of the ark LLM creation its common that engineers run the training data through the BPE to get the most accurate tokenaization for that specific data set.

#### Decoding


Now that we have the BPE algo trained lets use it. Suppose we have a set of tokens (integers from 0, vocab_size) what is the text that it decodes to?

In [55]:
vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

def decode(ids):
    byte_tokens = []

    for idx in ids:
        token_bytes = vocab[idx]
        byte_tokens.append(token_bytes)

    all_bytes = b"".join(byte_tokens)
    print(all_bytes)
    text = all_bytes.decode("utf-8", errors="replace")
    return text

print(decode([97, 32, 11, 111]))

b'a \x0bo'
a o


#### Encoder

In [56]:
def encode(text):
  # given a string, return list of integers (the tokens)
  tokens = list(text.encode("utf-8"))
  while len(tokens) >= 2:
    stats = get_stats(tokens)
    pair = min(stats, key=lambda p: merges.get(p, float("inf")))
    if pair not in merges:
      break # nothing else can be merged
    idx = merges[pair]
    tokens = merge(tokens, pair, idx)
  return tokens

print(encode("Hello world how are you"))

[72, 101, 108, 108, 111, 32, 119, 270, 108, 100, 32, 104, 111, 119, 32, 265, 256, 121, 111, 117]
